<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Launch Sites Locations Analysis with Folium**


Estimated time needed: **40** minutes


The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site, i.e., the initial position of rocket trajectories. Finding an optimal location for building a launch site certainly involves many factors and hopefully we could discover some of the factors by analyzing the existing launch site locations.


In the previous exploratory data analysis labs, you have visualized the SpaceX launch dataset using `matplotlib` and `seaborn` and discovered some preliminary correlations between the launch site and success rates. In this lab, you will be performing more interactive visual analytics using `Folium`.


## Objectives


This lab contains the following tasks:
- **TASK 1:** Mark all launch sites on a map
- **TASK 2:** Mark the success/failed launches for each site on the map
- **TASK 3:** Calculate the distances between a launch site to its proximities

After completed the above tasks, you should be able to find some geographical patterns about launch sites.


Let's first import required Python packages for this lab:


In [295]:
!pip3 install folium
!pip3 install wget
!pip3 install pandas

In [296]:
import folium
import wget
import pandas as pd

In [297]:
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon

If you need to refresh your memory about folium, you may download and refer to this previous folium lab:


[Generating Maps with Python](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/DV0101EN-3-5-1-Generating-Maps-in-Python-py-v2.0.ipynb)


## Task 1: Mark all launch sites on a map


First, let's try to add each site's location on a map using site's latitude and longitude coordinates


The following dataset with the name `spacex_launch_geo.csv` is an augmented dataset with latitude and longitude added for each site. 


In [303]:
# Download and read the `spacex_launch_geo.csv`
spacex_csv_file = wget.download('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv')
spacex_df=pd.read_csv(spacex_csv_file)

100% [................................................................................] 7710 / 7710

Now, you can take a look at what are the coordinates for each site.


In [305]:
#Update Launch_Site CCAFS there is only one but two names, the lat and log are changed too.
#spacex_df['Launch Site'] = spacex_df['Launch Site'].replace('CCAFS LC-40', 'CCAFS SLC-40')
#spacex_df['Lat'] = spacex_df['Lat'].replace('28.562302', '28.563197')
#spacex_df['Long'] = spacex_df['Long'].replace('-80.577356', '-80.576820')

# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


Above coordinates are just plain numbers that can not give you any intuitive insights about where are those launch sites. If you are very good at geography, you can interpret those numbers directly in your mind. If not, that's fine too. Let's visualize those locations by pinning them on a map.


We first need to create a folium `Map` object, with an initial center location to be NASA Johnson Space Center at Houston, Texas.


In [308]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=12)
site_map

We could use `folium.Circle` to add a highlighted circle area with a text label on a specific coordinate. For example, 


In [310]:
# Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
# Create a blue circle at NASA Johnson Space Center's coordinate with a icon showing its name
marker = folium.map.Marker(
    nasa_coordinate,
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

and you should find a small yellow circle near the city of Houston and you can zoom-in to see a larger circle. 


Now, let's add a circle for each launch site in data frame `launch_sites`


_TODO:_  Create and add `folium.Circle` and `folium.Marker` for each launch site on the site map


An example of folium.Circle:


`folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(...))`


An example of folium.Marker:


`folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'label', ))`


In [318]:

launch_sites_df.rename(columns={'Launch Site':'Launch_Site'}, inplace=True)
launch_sites_df


,Launch_Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


In [319]:
site_map = folium.Map(location=[37.0902, -95.7129], zoom_start=5)

# Agregar marcadores con nombres visibles como tooltips
for lat, lng, label in zip(launch_sites_df.Lat, launch_sites_df.Long, launch_sites_df.Launch_Site):
    folium.Marker(
        [lat, lng],
        tooltip=label,
        icon=None # Tooltip muestra el nombre del launch site
    ).add_to(site_map)

# Agregar círculos para marcar los launch sites visualmente
for lat, lng in zip(launch_sites_df.Lat, launch_sites_df.Long):
    folium.vector_layers.CircleMarker(
        [lat, lng],
        radius=5,
        color='yellow',
        fill=True,
        fill_color='blue',
        fill_opacity=0.6
    ).add_to(site_map)


# Mostrar el mapa
site_map


The generated map with marked launch sites should look similar to the following:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_markers.png">
</center>


Now, you can explore the map by zoom-in/out the marked areas
, and try to answer the following questions:
- Are all launch sites in proximity to the Equator line?
- Are all launch sites in very close proximity to the coast?

Also please try to explain your findings.


# Task 2: Mark the success/failed launches for each site on the map


Next, let's try to enhance the map by adding the launch outcomes for each site, and see which sites have high success rates.
Recall that data frame spacex_df has detailed launch records, and the `class` column indicates if this launch was successful or not


In [325]:

spacex_df.rename(columns={'Launch Site':'Launch_Site'}, inplace=True)
spacex_df.tail(10)

,Launch_Site,Lat,Long,class
46,KSC LC-39A,28.573255,-80.646895,1
47,KSC LC-39A,28.573255,-80.646895,1
48,KSC LC-39A,28.573255,-80.646895,1
49,CCAFS SLC-40,28.563197,-80.576820,1
50,CCAFS SLC-40,28.563197,-80.576820,1
51,CCAFS SLC-40,28.563197,-80.576820,0
52,CCAFS SLC-40,28.563197,-80.576820,0
53,CCAFS SLC-40,28.563197,-80.576820,0
54,CCAFS SLC-40,28.563197,-80.576820,1
55,CCAFS SLC-40,28.563197,-80.576820,0


Next, let's create markers for all launch records. 
If a launch was successful `(class=1)`, then we use a green marker and if a launch was failed, we use a red marker `(class=0)`


Note that a launch only happens in one of the four launch sites, which means many launch records will have the exact same coordinate. Marker clusters can be a good way to simplify a map containing many markers having the same coordinate.


Let's first create a `MarkerCluster` object


In [329]:
marker_cluster = MarkerCluster()


_TODO:_ Create a new column in `launch_sites` dataframe called `marker_color` to store the marker colors based on the `class` value


In [331]:

marker_color=[]

for value in spacex_df['class']:
    if value == 1:
        marker_color.append('green')
    else:
        marker_color.append('red')


spacex_df['MarkerColor'] = marker_color
spacex_df.tail()
# Apply a function to check the value of `class` column
# If class=1, marker_color value will be green
# If class=0, marker_color value will be red


,Launch_Site,Lat,Long,class,MarkerColor
51,CCAFS SLC-40,28.563197,-80.57682,0,red
52,CCAFS SLC-40,28.563197,-80.57682,0,red
53,CCAFS SLC-40,28.563197,-80.57682,0,red
54,CCAFS SLC-40,28.563197,-80.57682,1,green
55,CCAFS SLC-40,28.563197,-80.57682,0,red


In [332]:
spacex_df.groupby(['Launch_Site','Lat','Long'])['class'].count()

Launch_Site   Lat        Long       
CCAFS LC-40   28.562302  -80.577356     26
CCAFS SLC-40  28.563197  -80.576820      7
KSC LC-39A    28.573255  -80.646895     13
VAFB SLC-4E   34.632834  -120.610745    10
Name: class, dtype: int64

In [333]:
filtered_data=spacex_df[spacex_df['class']== 1]
filtered_data.groupby(['Launch_Site','Lat','Long'])['class'].count()

Launch_Site   Lat        Long       
CCAFS LC-40   28.562302  -80.577356      7
CCAFS SLC-40  28.563197  -80.576820      3
KSC LC-39A    28.573255  -80.646895     10
VAFB SLC-4E   34.632834  -120.610745     4
Name: class, dtype: int64

In [334]:
# Function to assign color to launch outcome
def assign_marker_color(launch_outcome):
    if launch_outcome == 1:
        return 'green'
    else:
        return 'red'
    
spacex_df['marker_color'] = spacex_df['class'].apply(assign_marker_color)
spacex_df.tail(10)

,Launch_Site,Lat,Long,class,MarkerColor,marker_color
46,KSC LC-39A,28.573255,-80.646895,1,green,green
47,KSC LC-39A,28.573255,-80.646895,1,green,green
48,KSC LC-39A,28.573255,-80.646895,1,green,green
49,CCAFS SLC-40,28.563197,-80.576820,1,green,green
50,CCAFS SLC-40,28.563197,-80.576820,1,green,green
51,CCAFS SLC-40,28.563197,-80.576820,0,red,red
52,CCAFS SLC-40,28.563197,-80.576820,0,red,red
53,CCAFS SLC-40,28.563197,-80.576820,0,red,red
54,CCAFS SLC-40,28.563197,-80.576820,1,green,green
55,CCAFS SLC-40,28.563197,-80.576820,0,red,red


_TODO:_ For each launch result in `spacex_df` data frame, add a `folium.Marker` to `marker_cluster`


In [336]:
marker_cluster1 = MarkerCluster()
site_map1 = folium.Map(location=[37.0902, -95.7129], zoom_start=5)
# Add marker_cluster to current site_map
site_map1.add_child(marker_cluster1)

# for each row in spacex_df data frame
# add pop-up text to each marker on the map
#latitudes = list(spacex_df.Lat)
#longitudes = list(spacex_df.Long)
#labels = list(spacex_df.class)

#for lat, lng, label in zip(latitudes, longitudes, labels):
 #   folium.Marker([lat, lng], popup=label).add_to(site_map)

# create a Marker object with its coordinate
# and customize the Marker's icon property to indicate if this launch was successed or failed, 

# Crear un objeto de MarkerCluster
#marker_cluster = MarkerCluster().add_to(site_map)


# Iterar sobre las filas del DataFrame
for index, record in spacex_df.iterrows():
    # Definir el icono dentro del bucle
    icon=folium.Icon(color='white', icon_color=record['MarkerColor'])
    # TODO: Create and add a Marker cluster to the site map
    # Crear el marcador y añadirlo al mapa
    marker = folium.Marker([record['Lat'],record['Long']], icon=icon, popup=str(record['Launch_Site'])).add_to(site_map1)
    marker_cluster1.add_child(marker)

# Añadir el marker_cluster al mapa
site_map1.add_child(marker_cluster1)
site_map1

Your updated map may look like the following screenshots:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster.png">
</center>


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster_zoomed.png">
</center>


From the color-labeled markers in marker clusters, you should be able to easily identify which launch sites have relatively high success rates.


# TASK 3: Calculate the distances between a launch site to its proximities


Next, we need to explore and analyze the proximities of launch sites.


Let's first add a `MousePosition` on the map to get coordinate for a mouse over a point on the map. As such, while you are exploring the map, you can easily find the coordinates of any points of interests (such as railway)


In [344]:
# Add Mouse Position to get the coordinate (Lat, Long) for a mouse over on the map
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


You can calculate the distance between two points on the map based on their `Lat` and `Long` values using the following method:


In [347]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

_TODO:_ Mark down a point on the closest coastline using MousePosition and calculate the distance between the coastline point and the launch site.


In [349]:
# find coordinate of the closet coastline
launch_site_lat=28.562302
launch_site_lon=-80.577356
coastline_lat= 28.56209
coastline_lon= -80.5678
distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)
print(f'{distance_coastline:.2f} km')

0.93 km


_TODO:_ After obtained its coordinate, create a `folium.Marker` to show the distance


In [351]:
# Create and add a folium.Marker on your selected closest coastline point on the map
#marker1 = folium.Marker([coastline_lat,coastline_lon], icon=icon, popup='coastline').add_to(site_map1)
# Display the distance between coastline point and launch site using the icon property 
# for example
distance_marker = folium.Marker(
    [coastline_lat,coastline_lon],
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_coastline),
        )
    ).add_to(site_map1)

site_map1.add_child(distance_marker)
site_map1

_TODO:_ Draw a `PolyLine` between a launch site to the selected coastline point


In [353]:
# Create a `folium.PolyLine` object using the coastline coordinates and launch site coordinate

coordinates=[
    [28.562302,-80.577356], #Launch_site
    [28.56209,-80.5678]    #coastline
]
lines=folium.PolyLine(locations=coordinates, weight=1)
site_map1.add_child(lines)

Your updated map with distance line should look like the following screenshot:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_distance.png">
</center>


_TODO:_ Similarly, you can draw a line betwee a launch site to its closest city, railway, highway, etc. You need to use `MousePosition` to find the their coordinates on the map first


A railway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/railway.png">
</center>


A highway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/highway.png">
</center>


A city map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/city.png">
</center>


In [363]:
# Create a marker with distance to a closest city, railway, highway, etc.
# Draw a line between the marker to the launch site
coordinates2=[
    [28.562302,-80.577356], #Launch_site
    [28.57112,-80.58541],    #railway
    [28.56248,-80.57051],    #highway
    [28.61381,-80.80753]    #city
]
lines=folium.PolyLine(locations=coordinates2, weight=1)
site_map1.add_child(lines)

In [364]:
railway_lat=28.57112
railway_lon=-80.58541
highway_lat=28.56248
highway_lon=-80.57051
city_lat=28.61381
city_lon=-80.80753
coast_lat=28.56209
coast_lon=-80.5678

In [365]:
distance_railway = calculate_distance(launch_site_lat, launch_site_lon, railway_lat, railway_lon)
distance_highway = calculate_distance(launch_site_lat, launch_site_lon, highway_lat, highway_lon)
distance_city = calculate_distance(launch_site_lat, launch_site_lon, city_lat, city_lon)
distance_coast = calculate_distance(launch_site_lat, launch_site_lon, coast_lat, coast_lon)
print(f'The distance to railway is: {distance_railway:.2f} km')
print(f'The distance to highway is: {distance_highway:.2f} km')
print(f'The distance to a city is: {distance_city:.2f} km')
print(f'The distance to the coast is: {distance_coast:.2f} km')

The distance to railway is: 1.26 km
The distance to highway is: 0.67 km
The distance to a city is: 23.20 km
The distance to the coast is: 0.93 km


After you plot distance lines to the proximities, you can answer the following questions easily:
- Are launch sites in close proximity to railways?
- Are launch sites in close proximity to highways?
- Are launch sites in close proximity to coastline?
- Do launch sites keep certain distance away from cities?

Also please try to explain your findings.


# Next Steps:

Now you have discovered many interesting insights related to the launch sites' location using folium, in a very interactive way. Next, you will need to build a dashboard using Ploty Dash on detailed launch records.


## Authors


[Yan Luo](https://www.linkedin.com/in/yan-luo-96288783/)


### Other Contributors


Joseph Santarcangelo


## Change Log


|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2021-05-26|1.0|Yan|Created the initial version|


Copyright © 2021 IBM Corporation. All rights reserved.
